In [2]:
from langgraph.graph import StateGraph ,START,END
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated, Literal
from pydantic import BaseModel, Field
from  dotenv import load_dotenv
import operator

In [3]:
load_dotenv()
model = ChatGroq(
    model="openai/gpt-oss-120b", 
)

In [4]:
class SentimentSchema(BaseModel):
    
    sentiment: Literal["postive","negative"] = Field(description="Sentiment of the review")

In [5]:
class ReviewState(TypedDict):
    review: str
    sentiment: Literal["postive","negative"]
    diagnosis: dict
    response: str

In [6]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX","performance","Bug","Support","Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry","frustrated","disappointed"] = Field(description='The emtional tine expressed by the user')
    urgency : Literal["low",'medium','high'] = Field(description="How urgent or critical the issue appears to be")
    
    

In [7]:
structure_model1 = model.with_structured_output(SentimentSchema)
structure_model2 = model.with_structured_output(DiagnosisSchema)

In [10]:
prompt = 'what is the sentiment of the following review -  the software too good'
structure_model1.invoke(prompt).sentiment

'postive'

In [15]:
def find_sentiment(state: ReviewState):

    prompt= f'for the following review find out the sentiment \n {state['review']}'
    sentiment = structure_model1.invoke(prompt).sentiment
    
    return {'sentiment': sentiment}


def check_sentiment(state: ReviewState) -> Literal['positive_response','run_diagnosis']:
    
    if state['sentiment'] == "postive":
        return 'positive_response'
    else:
        return "run_diagnosis"


def positive_response(state: ReviewState):
    
    prompt  = f'Writea warn thank you  message  in respnse  to this  review \n\n "{state['review']}" \n also kindly ask the user to feedback on our website'
    response = model.invoke(prompt).content
    return {'response': response}


def run_diagnosis(state: ReviewState):
    prompt = f""" Diagnose this negative review: \n\n {state['review']} \n
    Return issue_type, tone and uegency.
    """
    response = structure_model2.invoke(prompt)
    return {'diagnosis': response.model_dump()}

def negative_response(state: ReviewState):
    diagnosis = state['diagnosis']
    prompt = f""" you are a support assistant.
    The user had a '{diagnosis['issue_type']}' issue , sounded '{diagnosis['tone']}'  and marked unrecy as 
    {diagnosis['urgency']}.
    Write an empathetic helpful resolution message.
    """
    
    response = model.invoke(prompt).content
    return {'response': response}

In [16]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment',find_sentiment) 
graph.add_node('positive_response',positive_response)
graph.add_node('run_diagnosis',run_diagnosis)
graph.add_node('negative_response',negative_response)

graph.add_edge(START,'find_sentiment')
graph.add_conditional_edges('find_sentiment',check_sentiment) 
graph.add_edge('positive_response',END)
graph.add_edge('run_diagnosis','negative_response')
graph.add_edge('negative_response',END)

workflow= graph.compile()


In [17]:
intial_state={
    'review': "I’ve been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality."
}
workflow.invoke(intial_state) 

{'review': 'I’ve been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality.',
 'sentiment': 'negative',
 'diagnosis': {'issue_type': 'Bug', 'tone': 'angry', 'urgency': 'high'},
 'response': 'Hi\u202f[Customer Name],\n\nI’m really sorry to hear that you’ve run into this bug and that it’s caused such frustration. I completely understand how upsetting it can be when something isn’t working as expected, especially when you’re trying to get things done.\n\n**Here’s what we can do right away:**\n\n1. **Confirm the details** – Could you please share the exact steps you took before the issue occurred, along with any error messages or screenshots you have? This information helps us pinpoint the problem faster.\n\n2. **Immediate workaround** – While we investigate, you might try the following temporary fix: *[brief, relevant workar